In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [4]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])
region = ee.Geometry.Rectangle([91.34461440693306, 22.72071117953768, 91.53138198505806, 22.854601319589566])

map.centerObject(region, 10)
map.add_basemap('HYBRID')
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')


map.default_style = {'cursor': 'crosshair'}


map #just testing if everything is working

Map(center=[22.78767247055464, 91.43799819599587], controls=(WidgetControl(options=['position', 'transparent_b…

In [7]:
#finding the flood

floodImage = ee.ImageCollection('COPERNICUS/S1_GRD') \
  .filterBounds(region) \
  .filterDate('2024-06-01', '2024-09-30') \
  .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
  .median() \
  .clip(region)

#removing the noise
smoothed = floodImage.select('VH').focal_median(50, 'circle', 'meters')
waterThreshold = -18
floodMask = smoothed.lt(waterThreshold) #creating a binary image based on threshold

map.remove_layer('Point Layer') 
map.remove_layer('Detected Water') 
map.remove_layer("Region Layer")
map.addLayer(floodMask.selfMask(), {'palette': 'blue'}, 'Detected Water')
map

Map(bottom=228368.0, center=[22.77364906658098, 91.43989562988281], controls=(WidgetControl(options=['position…

In [8]:
#now loading the google dymamic world
landCover = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
  .filterBounds(region) \
  .filterDate('2024-01-01', '2024-06-01') \
  .mode() \
  .clip(region)


cropMask = landCover.select('label').eq(4) #4 is the class for crops in dynamic world

map.remove_layer('Detected Water')
map.addLayer(cropMask.selfMask(), {'palette': 'green'}, 'Crops Before Flood')
map

Map(bottom=456394.0, center=[22.78694384438262, 91.42770767211915], controls=(WidgetControl(options=['position…

In [10]:
#now at the same time finding the crops that are flooded

floodedCrops = floodMask.And(cropMask)
map.addLayer(floodedCrops.selfMask(), {'palette': 'red'}, 'Flooded Crops')

map

Map(bottom=456394.0, center=[22.78694384438262, 91.42770767211915], controls=(WidgetControl(options=['position…

In [11]:
#calculating the area of the flooded crops

areaImage = ee.Image.pixelArea().multiply(floodedCrops)
stats = areaImage.reduceRegion(
  reducer=ee.Reducer.sum(),
  geometry=region,
  scale=10,
  maxPixels=1e9
)

damageSqKm = ee.Number(stats.get('area')).divide(1e6)
print("flooded area (sq km):", damageSqKm.getInfo())

flooded area (sq km): 43.57763035466778
